# Paper 14 — VoiceStudio Native Mac Voice Evaluation

## Abstract
This study evaluates a pinned native VoiceStudio backend as an additional local conversational voice engine alongside the preserved Money/F5 pipeline. It distinguishes API execution, output integrity, device evidence, and subjective listening quality. No quality winner or commercial clearance is assumed.

**Status: bounded API and processing experiments executed; listening quality and complete Dia dialogue delivery remain unverified.**

## Research question and hypotheses
Can supported Mac engines produce a source-traceable two-speaker exchange through the local API, and does existing mastering/enhancement improve listening without damaging intelligibility? Successful HTTP responses alone do not establish either conversational quality or usefulness.

## Protocol
1. Pin backend and model revisions; inspect requirements and terms without accepting agreements.
2. Prepare selected assets in isolated storage; verify sizes and available weight hashes.
3. Generate one real utterance using an explicit engine and raw application effects.
4. Generate the same bounded exchange with Dia, full VoxCPM2 where feasible, and the existing F5 baseline.
5. Preserve originals; compare justified mastering/enhancement variants at matched loudness.
6. Record actual settings, output integrity, timing, memory, device observations, failures, and listening limitations.

## Isolation and network observations
VoiceStudio v0.5.7 is pinned to `beeaae08204f0de5f49d4a087ef2b0ea8fecfcb9`, bound to loopback port 5196. Its environment, cache and data are separate from Money/F5. Startup ONNX telemetry was observed as a firewall attempt; the server was stopped and restarted with `ORT_DISABLE_TELEMETRY=1`. Inference is configured with Hugging Face offline flags. Public model assets are explicitly downloaded via bounded HTTPS requests. No firewall rules or agreements have been accepted by this study. See the versioned network note for evidence and limits.


In [1]:
from pathlib import Path
import json, urllib.request
ROOT = Path('/Users/x/projects/swarms')
STUDY = ROOT / 'knowledge/research/paper-14-study'
def read_json(name):
    return json.loads((STUDY/name).read_text())
downloads = read_json('downloads.json')
print(downloads['stage'])
for model in downloads['models']:
    print(model['repo'], model['revision'], model['bytes'], 'verified bytes')

complete
mlx-community/Dia-1.6B de4fa8c178ca5cc4e9d884b55b03fcfaa0995162 6444684602 verified bytes
mlx-community/descript-audio-codec-44khz 91dff27c81edbb0992be17ac51d2c7d515411a32 306645112 verified bytes
openbmb/VoxCPM2 32279effe8c19989596f05d353d1447f51d9e915 4960730347 verified bytes


In [2]:
# Read-only local API check. This does not prove model inference.
with urllib.request.urlopen('http://127.0.0.1:5196/health', timeout=15) as response:
    print(json.load(response))

{'status': 'ok', 'device': 'mps', 'version': '0.5.7'}


## Results, failures, and interpretation
Real generation results appear below. Downloads, package readiness and health are reported separately from inference. The model cards report permissive licenses for the selected assets; that is not a complete commercial or voice-rights assessment.

## Listening limitations
Automated waveform checks cannot establish naturalness, emotional delivery, pronunciation, or speaker consistency. Listening judgments must be identified as such, and cross-engine casting differences must remain explicit.


## Fixed conversation and preserved F5 reference
The following recordings were generated and verified in Paper 13, rather than regenerated here. The exact spoken text is the comparison script. F5 uses its existing emotion reference recordings; Dia uses its native two-speaker tags. These are different casting mechanisms, so this is a practical workflow comparison rather than an isolated test of model architecture. Original, mastered and enhanced files are preserved separately.

In [3]:
from IPython.display import Audio, display
baseline = read_json('baseline.json')
for turn in baseline:
    print(turn['turn'], turn['text'])
    display(Audio(filename=turn['conditions']['voice']['file']))

0 I keep collecting ideas, but none of them feels ready to begin.


1 Then let’s choose one small thing we can finish together, today.


2 Just one? A light, a path, and a first step?


3 Exactly. We don’t need the whole bridge. We need somewhere safe to start.


## Native API experiment
The helper below saves immutable per-run requests, response headers, native WAVs, and integrity measurements. It does not retry generation automatically. Run only after selected models are completely prepared; no model inference result is implied by this cell being present.

In [4]:
import importlib.util
spec = importlib.util.spec_from_file_location('api_experiment', STUDY/'api_experiment.py')
experiment = importlib.util.module_from_spec(spec)
spec.loader.exec_module(experiment)
# First utterance (explicitly invoke once after setup):
# result = experiment.generate('dia-utterance-01', '[S1] One small step can turn an idea into something real.')
# display(Audio(filename=result['audio']['path']))

## First measured API utterance
The first Dia request returned HTTP 200 and a 4.123-second, mono 24 kHz PCM WAV. The integrity check confirms finite, non-silent samples, not pronunciation or naturalness. End-to-end time was 61.58 seconds. The upstream render trace attributes 11.33 seconds to synthesis and 49.77 seconds to an unsuccessful AudioSeal download. Consequently this first output did **not** receive the intended invisible watermark. It remains explicitly identified here as synthetic audio.

The observation wrapper recorded MLX default GPU device, 6.75 GB active allocation and 10.58 GB peak allocation after this request. These are framework allocation observations, not a kernel-level execution trace. Torch forward hooks recorded zero calls for this MLX synthesis. A pinned AudioSeal generator was subsequently prepared in the isolated cache. Later VoxCPM calls successfully loaded the generator and ran the watermark stage locally; this does not retroactively mark the first Dia recordings.


In [5]:
first = read_json('runs/dia-utterance-01/result.json')
print(json.dumps(first, indent=2))
display(Audio(filename=first['audio']['path']))

{
  "started_unix": 1793363968.951423,
  "endpoint": "http://127.0.0.1:5196/generate",
  "status": 200,
  "headers": {
    "date": "Fri, 30 Oct 2026 12:39:28 GMT",
    "server": "uvicorn",
    "x-audio-id": "2822bffc",
    "x-gen-time": "61.14",
    "x-audio-path": "2822bffc.wav",
    "x-seed": "42",
    "x-audio-duration": "4.12",
    "content-length": "296918",
    "content-type": "audio/wav",
    "x-omnivoice-backend": "0.5.7"
  },
  "elapsed_seconds": 61.5760760307312,
  "audio": {
    "path": "/Users/x/projects/swarms/knowledge/research/paper-14-study/runs/dia-utterance-01/native.wav",
    "sha256": "fa631fb1e109b59b53f0dd6f46b695d40cd6593f708ccd01123749268143c38e",
    "sample_rate": 24000,
    "frames": 98958,
    "channels": 1,
    "duration_seconds": 4.12325,
    "finite": true,
    "peak": 0.9992251396179199,
    "rms": 0.18658331394346442
  },
  "integrity_passed": true
}


## Existing Money processing applied to native Dia audio
Balanced mastering and neural enhancement (32 evaluations, strength 0.1) both completed on the first Dia sample. Independent checks verified source hashes, finite audio, the configured loudness/peak limits, matched-comparison loudness within 0.25 LU, neural forward calls, and unchanged Money settings/latest pointers. This establishes integration, not a subjective quality improvement.


In [6]:
processed = read_json('processing/dia-utterance-01/result.json')['outputs']
for stage in ('master','enhance'):
    print(stage, 'matched before / matched after')
    display(Audio(filename=processed[stage]['before']))
    display(Audio(filename=processed[stage]['after']))
print(read_json('processing/dia-utterance-01/independent-verification.json'))

master matched before / matched after


enhance matched before / matched after


{'master': {'seconds': 4.12325, 'sample_rate': 48000, 'peak': 0.8179835081100464, 'sha256': 'd6f3e4c574534be244f435dc5f5a595fc50a7385905d4d209b655ee68fa969f2', 'independent_matched_difference_lu': 0.0, 'independent_master_measurement': {'input_i': '-16.0', 'input_tp': '-1.7', 'meter': 'FFmpeg ebur128'}}, 'enhance': {'seconds': 4.123265306122449, 'sample_rate': 44100, 'peak': 0.7100859880447388, 'sha256': '5a018389f4f2c350f47c921b07fffb0b780762a0e5a076c1cf8a2e20764eb837', 'independent_matched_difference_lu': 0.0}, 'money_settings_and_latest_pointers_unchanged': True}


## Conversation results: full VoxCPM2, MLX Dia, and preserved F5
All four VoxCPM2 calls returned non-silent finite WAVs at 48 kHz. Each used seed 42, 10 inference steps, application effects set to raw, and an existing ESD reference from the corresponding F5 condition. The native adapter trims reference silence and caps reference length at 30 seconds. Written delivery instructions select controllable cloning, rather than transcript continuation. This is the full PyTorch engine, not the separate MLX VoxCPM conversion.

Dia received the whole tagged conversation in one request, with its native speaker identities. Its output is only 9.92 seconds long. That is substantially shorter than the other conditions, so complete text delivery and turn separation **remain unverified**; waveform integrity is insufficient to certify the exchange. This difference must not be interpreted as better speed or conversational quality.

VoxCPM2 and F5 turn files are joined with 200 ms gaps for convenient playback. Dia's internally generated timing is preserved. Speaker casting, batching, warm-up, and processing differ. The cold VoxCPM2 request also overlapped a separate Money enhancement task briefly. These timings are observations, not a controlled throughput benchmark. None of these conditions establishes commercial voice rights.


In [7]:
print(json.dumps(read_json('measured-results.json'), indent=2))
print('Dia: native two-speaker generation; completeness needs listening review')
display(Audio(filename=str(STUDY/'runs/dia-dialogue-01/native.wav')))
for item in read_json('assembled-dialogues.json'):
    print(item['engine'], 'original exchange')
    display(Audio(filename=item['file']))

[
  {
    "run": "dia-utterance-01",
    "seconds": 4.12325,
    "api_seconds": 61.5760760307312,
    "sample_rate": 24000,
    "peak": 0.9992251396179199,
    "integrity_passed": true
  },
  {
    "run": "dia-dialogue-01",
    "seconds": 9.916625,
    "api_seconds": 21.86178684234619,
    "sample_rate": 24000,
    "peak": 1.0,
    "integrity_passed": true
  },
  {
    "run": "voxcpm-turn-0-01",
    "seconds": 4.32,
    "api_seconds": 50.9280960559845,
    "sample_rate": 48000,
    "peak": 0.27540671825408936,
    "integrity_passed": true
  },
  {
    "run": "voxcpm-turn-1-01",
    "seconds": 5.12,
    "api_seconds": 5.305459976196289,
    "sample_rate": 48000,
    "peak": 0.8355463743209839,
    "integrity_passed": true
  },
  {
    "run": "voxcpm-turn-2-01",
    "seconds": 4.48,
    "api_seconds": 4.44834303855896,
    "sample_rate": 48000,
    "peak": 0.22764968872070312,
    "integrity_passed": true
  },
  {
    "run": "voxcpm-turn-3-01",
    "seconds": 4.16,
    "api_seconds": 4.2

voxcpm original exchange


f5 original exchange


## Reproduce native conversation generation
Use a fresh run prefix to preserve existing evidence. The backend launcher is `services/voicestudio/launch.py`; its environment and model assets must first be prepared as recorded in the setup manifests. The function below performs real API generation when explicitly invoked; it does not silently regenerate audio when opening the notebook.


In [8]:
def generate_conversation(prefix, engine):
    plan = read_json('dialogue-plan.json')
    if engine == 'mlx-audio':
        return [experiment.generate(prefix, plan['dia_text'], engine=engine)]
    if engine == 'voxcpm2':
        return [experiment.generate(f'{prefix}-turn-{i}', turn['text'], engine=engine,
                    instruct=turn['voxcpm_proposed_instruction'],
                    ref_audio=turn['reference_audio'], ref_text=turn['reference_transcript'],
                    num_step=10) for i, turn in enumerate(plan['turns'])]
    raise ValueError('Choose the evaluated mlx-audio or voxcpm2 engine')
# Explicit examples, with fresh names:
# generate_conversation('dia-new-comparison', 'mlx-audio')
# generate_conversation('voxcpm-new-comparison', 'voxcpm2')

## Mastered and enhanced VoxCPM2 exchange
These processing conditions use the same existing Money engines. Compare each matched before/after pair rather than assuming a louder output is better. The independent verifier checks waveform, source preservation, configured loudness limits, matched loudness and neural execution evidence. It does not rate listening quality.


In [9]:
vox_processed = read_json('processing/voxcpm-dialogue-01/result.json')['outputs']
for stage in ('master','enhance'):
    print(stage, 'matched before / matched after')
    display(Audio(filename=vox_processed[stage]['before']))
    display(Audio(filename=vox_processed[stage]['after']))
print(read_json('processing/voxcpm-dialogue-01/independent-verification.json'))

master matched before / matched after


enhance matched before / matched after


{'master': {'seconds': 18.68, 'sample_rate': 48000, 'peak': 0.8432407379150391, 'sha256': 'a5fd08f5c9c2f9cf6d0fe463f03cdf2147f66a0c5df250ab1002a90a288835de', 'independent_matched_difference_lu': 0.10000000000000142, 'independent_master_measurement': {'input_i': '-15.9', 'input_tp': '-1.5', 'meter': 'FFmpeg ebur128'}}, 'enhance': {'seconds': 18.68, 'sample_rate': 44100, 'peak': 0.8351937532424927, 'sha256': 'a03d8d45732897451897d069497353505aef80cc819cdd440b35b8853fa01754', 'independent_matched_difference_lu': 0.0}, 'money_settings_and_latest_pointers_unchanged': True}


## Device evidence and memory
A separate repeated VoxCPM2 turn used the same pinned checkpoint, reference and request controls with observational Torch hooks enabled. The sidecar recorded at least 68,000 module-forward calls with both parameter and input tensor devices on `mps:0`, approximately 10.59 GB MPS allocation, 11.64 GB driver allocation, and a process maximum RSS of 15.33 GB at the last sample. The sampling interval means the forward count is a lower bound and these are sampled memory observations. Hooks can add overhead. This probe is reported separately from the original conversation timings.

Dia's MLX allocation records are framework observations, whereas VoxCPM2 and preserved F5 references provide module-level device observations. They are different measurement boundaries and cannot be compared as identical telemetry.


In [10]:
print(read_json('voxcpm-device-evidence.json'))
print('Repeated device probe:')
probe = read_json('runs/voxcpm-device-probe-01/result.json')
print(probe['elapsed_seconds'], probe['audio'])
display(Audio(filename=probe['audio']['path']))

{'pid': 99227, 'started_unix': 1793364389.40094, 'forward_calls': 68000, 'observed_parameter_devices': ['mps:0'], 'observed_tensor_devices': ['mps:0'], 'observed_unix': 1793364404.93401, 'process_maxrss_bytes': 15326953472, 'mps_allocated_bytes': 10586457600, 'mps_driver_bytes': 11643994112}
Repeated device probe:
16.57725191116333 {'path': '/Users/x/projects/swarms/knowledge/research/paper-14-study/runs/voxcpm-device-probe-01/native.wav', 'sha256': '6639ac5162b31aac34fe0d53dd99238419b9484b003b9af17ce9ccf54c75896f', 'sample_rate': 48000, 'frames': 245760, 'channels': 1, 'duration_seconds': 5.12, 'finite': True, 'peak': 0.8355463743209839, 'rms': 0.1200018785333333}


## Conclusions and next decision
The pinned native backend can serve actual local speech through its API. Full VoxCPM2 is feasible on this Mac and generated the four-turn reference-conditioned exchange; subsequent warm turns took roughly 4–5 seconds each. MLX Dia generated both a short utterance and a tagged conversation, but the short duration of the latter raises a completeness question that this study does not resolve. Both native outputs can pass through the preserved Money mastering/enhancement engines with independently checked signal and provenance integrity.

These results support keeping VoiceStudio as an **additional candidate engine**. They do not justify replacing the user-approved F5/mastering/enhancement standard, declaring a subjective winner, or claiming commercial clearance. Human listening is still needed for text accuracy, speaker continuity, emotional delivery, artifacts and preference. No listening ratings are invented here. A future transcript assessment could test omissions, but no extra ASR model was downloaded in this bounded study.

## Reproducibility and provenance
- Backend: [VoiceStudio v0.5.7](https://github.com/debpalash/VoiceStudio/tree/beeaae08204f0de5f49d4a087ef2b0ea8fecfcb9).
- Dia: [pinned model](https://huggingface.co/mlx-community/Dia-1.6B/tree/de4fa8c178ca5cc4e9d884b55b03fcfaa0995162), plus its pinned DAC codec.
- VoxCPM2: [pinned model](https://huggingface.co/openbmb/VoxCPM2/tree/32279effe8c19989596f05d353d1447f51d9e915), official managed package 2.0.3 in a separate environment.
- AudioSeal: [pinned auxiliary generator](https://huggingface.co/facebook/audioseal/tree/3c19eba53390776cf2cc9ed5f6c9ac67ce72ecba).

The study directory records model cards, requirements and license metadata, revisions, hashes, package inventories, request/response records, preserved baseline provenance, processing requests/reports and independent verification. License metadata and an ungated download are not a full rights assessment. No gated agreements were accepted. The API is loopback-only; its local-admin behavior is not presented as protection from other processes on the same computer.
